# Nemotron-3.5 Super VL Finance LoRA Fine-Tuning using Megatron Bridge

Data prep, checkpoint conversion, LoRA fine-tuning, then merge and try-out, using [NeMo Megatron-Bridge](https://github.com/NVIDIA-NeMo/Megatron-Bridge).

One adapter learns to read values off financial document images, calculate from extracted evidence, and write small Python programs for financial questions. Training uses text and images together.

## Hardware

**Minimum:** one node with 8 GPUs of 80 GB or more, and about 800 GB of disk. Measured on H200 (141 GB), training one epoch:

| GPUs | TP / PP / EP | Peak memory/GPU | One epoch |
| --- | --- | --- | --- |
| 4 | 4 / 1 / 4 | ~95 GB | ~7.5 h |
| 8 (default) | 2 / 1 / 8 | ~98 GB | ~1.75 h |
| 8 | 4 / 1 / 8 | ~66 GB | ~3.5 h |

Memory figures are PyTorch allocations; `nvidia-smi` includes cached memory and can show much higher usage. On 80 GB GPUs (H100) use `TRAIN_TP=4`. The 4-GPU layout covers training only; the try-out was verified on 8 GPUs.

## Prerequisites

- **Model:** [`nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-BF16), about 250 GB. Export a Hugging Face token with access to it as `HF_TOKEN` before launching the container.
- **Container:** `nvcr.io/nvidian/nemo:26.10.rc4` (Megatron-Bridge 0.7.0). Every step runs inside it.
- **`config.env`** in this directory: set `WORKSPACE` to an absolute path with about 800 GB free; everything is written under it. The model, the container, the GPU layout (`GPUS_PER_NODE`, `TRAIN_TP`, `TRAIN_PP`, `TRAIN_EP`), and the run length are set there too, each with a one-line comment.

## Launching the container

Launch the container with this directory and your workspace mounted, then start Jupyter inside it:

```bash
source config.env
docker run --gpus all -it --rm -e HF_TOKEN \
  -v "$PWD:$PWD" -v "$WORKSPACE:$WORKSPACE" -w "$PWD" \
  --shm-size=16g --net=host --ipc=host \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  "$CONTAINER_IMAGE" jupyter lab --allow-root --no-browser
```

On a Slurm cluster, run the same steps as jobs instead: see the README.

The logs are noisy; [Expected friction](SKILL.md#expected-friction) lists the warnings that are harmless.

---
## Configuration

`config.env` holds the paths, the GPU layout, and the run length; every step reads it. Edit it there, not here; this cell shows what the steps will use.

In [ ]:
%%bash
cat config.env

---
## Step 1: Download the Model

Downloads the checkpoint (about 250 GB) to `$WORKSPACE/model`; every later step reads this copy. Re-running resumes or skips.

In [ ]:
%%bash
bash steps/download.sh

---
## Step 2: Data Preparation

Builds the training data from two public datasets, at pinned revisions:

- [TAT-DQA](https://huggingface.co/datasets/next-tat/TAT-DQA): questions about pages of company annual reports. For span questions the model sees a crop of the page around the evidence and reads the answer off the image; for arithmetic and count questions it gets the text of that crop and calculates.
- [BizBench](https://huggingface.co/datasets/kensho/bizbench): finance questions answered by writing a short Python program, as text.

Each training row is one conversation, with the model's `<think>` reasoning before the answer. Both datasets' published splits are pooled, and one in ten source documents (an annual report, or a BizBench context) is held out as this example's own test set. Takes 10 to 15 minutes on CPU.

Each dataset is one small class in `dataset_<name>.py`; see the README to add your own.

In [ ]:
%%bash
bash steps/dataprep.sh

### Sanity Check

Expect 22,222 training and 2,943 test rows. Look at one image example the way the model sees it.

In [ ]:
import json
import subprocess
from pathlib import Path

from IPython.display import display
from PIL import Image

workspace = subprocess.check_output(["bash", "-c", 'source config.env && printf %s "$WORKSPACE"'], text=True)
data_dir = Path(workspace) / "dataprep"
rows = [json.loads(line) for line in open(data_dir / "train.jsonl")]
print("Training rows:", len(rows), "  test rows:", sum(1 for _ in open(data_dir / "test.jsonl")))

# One example, preferably with an image, the way the model sees it.
user, assistant = next((r["messages"] for r in rows if r["messages"][0]["content"][0]["type"] == "image"), rows[0]["messages"])
for part in user["content"]:
    if part["type"] == "image":
        display(Image.open(part["image"]))
    else:
        print(part["text"])
print("\n--- target ---\n" + assistant["content"][0]["text"])

---
## Step 3: Model Conversion

Megatron-Bridge trains from its own checkpoint format, so convert the Hugging Face weights once. Uses 4 GPUs, about 30 minutes, most of it with no output while the weights load.

The result reshards on load, so you can convert once and train on any GPU layout. Re-running is a no-op once the checkpoint exists.

In [ ]:
%%bash
bash steps/convert.sh

### Sanity Check

In [ ]:
%%bash
source config.env
test -e "$WORKSPACE/base/latest_checkpointed_iteration.txt" || { echo "No converted checkpoint at $WORKSPACE/base"; exit 1; }
ls "$WORKSPACE/base"

---
## Step 4: LoRA Fine-Tuning

LoRA trains small adapter matrices next to the frozen model weights, so it needs far less memory than full fine-tuning. The shipped Megatron-Bridge recipe supplies the model-specific config: adapters on the language model's attention, Mamba, and expert projections, with the vision encoder frozen. `train.py` points it at your data, checkpoint, and GPU layout (`TRAIN_TP`, `TRAIN_PP`, `TRAIN_EP` in `config.env`).

Training runs `ceil(training rows / 16) × EPOCHS` steps (16 examples per step, from the recipe), capped by `MAX_STEPS`; `train.py` prints the resolved settings first.

For a quick smoke run first, set `MAX_STEPS=3` and `EXPERIMENT_NAME=smoke` in `config.env`; then clear `MAX_STEPS` and restore the name before the real run.

> Expect 5 to 12 minutes before iteration 1 appears, much of it silent while loading the base checkpoint.

In [ ]:
%%bash
bash steps/train.sh

### Sanity Check

Confirm a LoRA adapter checkpoint was saved.

In [ ]:
%%bash
source config.env
RUN_DIR=$WORKSPACE/trained/$EXPERIMENT_NAME
test -e "$RUN_DIR/latest_checkpointed_iteration.txt" || { echo "No saved adapter under $RUN_DIR"; exit 1; }
ls "$RUN_DIR"

---
## Step 5: Merge the LoRA Adapter and Export to Hugging Face

Merge the adapter back into the base weights and write a standard Hugging Face checkpoint to `$WORKSPACE/trained/$EXPERIMENT_NAME/merged_hf`, ready for inference or upload. Uses 4 GPUs, 15 to 25 minutes, with long stretches of no output.

In [ ]:
%%bash
bash steps/merge.sh

### Sanity Check

In [ ]:
%%bash
source config.env
ls "$WORKSPACE/trained/$EXPERIMENT_NAME/merged_hf"

---
## Step 6: Try the Fine-Tuned Model with vLLM

Answer the first eight held-out questions, images and text mixed, and print each answer next to the reference. Then ask the base model the same questions to see what fine-tuning changed. Each model takes 5 to 50 minutes, almost all of it loading the weights; how long depends on how fast your filesystem reads them.

In [ ]:
%%bash
bash steps/try.sh

In [ ]:
%%bash
bash steps/try.sh base

## Done

Your fine-tuned model is a standard Hugging Face checkpoint at `$WORKSPACE/trained/$EXPERIMENT_NAME/merged_hf`.